# MolSysMT rigid-fragment consumption

Four original Vina cases, with explicit selected cuts and source-to-retained atom
maps. The captured baseline is software behavior from DockingMT `be39d74`, not
scientific ground truth. Charges, types and chemical torsion eligibility remain
provisional. See `rigid_fragment_consumption.md` and DockingMT #6/#33.
The existing RDKit source bridge is retained for unsupported original SDF dialects.


In [1]:
import hashlib
import json
from pathlib import Path
import subprocess
import sys

import molsysmt as msm
from rdkit import Chem
from dockingmt.preparation import prepare_ligand

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "devguide/validation/data/fragments/legacy_trees.json").is_file())
baseline = json.loads((root / "devguide/validation/data/fragments/legacy_trees.json").read_text())
provider_root = Path(msm.__file__).resolve().parents[1]
provider_source = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=provider_root, text=True).strip()
print(json.dumps({"python": sys.version.split()[0], "interpreter": sys.executable,
                  "provider_source": provider_source,
                  "provider_runtime_version": msm.__version__,
                  "baseline_source": baseline["dockingmt_source"]}, indent=2))


{
  "python": "3.14.7",
  "interpreter": "/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python",
  "provider_source": "bd65456e0ca994f4a92800bda5d23325c999a673",
  "provider_runtime_version": "0.22.4+24.g42b487869",
  "baseline_source": "be39d74b5d437b958d0dde16e284453ecc7eef28"
}


## Compare exact prior output and expose the atom maps

MolSysMT partitions the complete source chemical graph. Only memberships are
projected onto retained atoms. ROOT orientation stays in DockingMT. Each PDBQT
position maps first to the retained axis, then to the original source index.


In [2]:
comparisons = []
maps = []
prepared_cases = []
for case in baseline["cases"]:
    suffix = "H" if case["case"].startswith("5x72") else ""
    source_path = root / "tests/data/vina_torsions" / (case["case"] + suffix + ".sdf")
    reference_path = root / "tests/data/vina_torsions" / (case["case"] + ".pdbqt")
    assert hashlib.sha256(source_path.read_bytes()).hexdigest() == case["source_sdf_sha256"]
    assert hashlib.sha256(reference_path.read_bytes()).hexdigest() == case["reference_pdbqt_sha256"]
    molecule = Chem.SDMolSupplier(str(source_path), removeHs=False)[0]
    assert molecule is not None
    source = msm.convert(molecule, to_form="molsysmt.MolSys")
    prepared = prepare_ligand(source, selection="all", active_torsion_bonds=case["active_bonds"])
    retained = prepared.metadata["retained_atom_indices"]
    assert retained == case["retained_atom_indices"]
    assert prepared.pdbqt_atom_indices == case["pdbqt_atom_indices"]
    digest = hashlib.sha256(prepared.to_pdbqt().encode()).hexdigest()
    assert digest == case["generated_pdbqt_sha256"]
    pairs = msm.get(source, element="bond", bonded_atom_pairs=True,
                    chemical_state="structure", structure_indices=0)
    identifiers = msm.get(source, element="bond", index=True,
                          chemical_state="structure", structure_indices=0)
    pair_to_bond = {frozenset(map(int, pair)): int(index)
                    for pair, index in zip(pairs, identifiers)}
    report = msm.topology.get_rigid_fragments(
        source, bond_indices=[pair_to_bond[frozenset(pair)] for pair in case["active_bonds"]],
        chemical_state="structure", structure_indices=0)
    offsets = report["fragment_offsets"]
    packed = report["fragment_atom_indices"]
    retained_set = set(retained)
    fragments = [[int(atom) for atom in packed[start:stop] if int(atom) in retained_set]
                 for start, stop in zip(offsets[:-1], offsets[1:])]
    comparisons.append({"case": case["case"], "branches": prepared.torsion_dof,
                        "retained_atoms": len(retained), "fragment_sizes": sorted(map(len, fragments)),
                        "prior_bytes_match": True, "prior_permutation_matches": True,
                        "pdbqt_sha256": digest})
    maps.append({"case": case["case"], "retained_to_source": retained,
                 "pdbqt_to_retained": prepared.pdbqt_atom_indices,
                 "pdbqt_to_source": [retained[i] for i in prepared.pdbqt_atom_indices],
                 "projected_fragment_source_indices": fragments})
    prepared_cases.append(prepared)
print(json.dumps(comparisons, indent=2))


[04:37:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


[
  {
    "case": "1iep_ligand",
    "branches": 7,
    "retained_atoms": 40,
    "fragment_sizes": [
      1,
      2,
      4,
      6,
      6,
      6,
      7,
      8
    ],
    "prior_bytes_match": true,
    "prior_permutation_matches": true,
    "pdbqt_sha256": "0f8404bb788564fb3dfbfc537651d08534c28b34ce338ca1e6794304452f95ee"
  },
  {
    "case": "1s63_ligand",
    "branches": 6,
    "retained_atoms": 29,
    "fragment_sizes": [
      1,
      1,
      2,
      5,
      6,
      7,
      7
    ],
    "prior_bytes_match": true,
    "prior_permutation_matches": true,
    "pdbqt_sha256": "922dde20fe6d1fe0435aa8f1dd0e09658e186c17fa371fb5665a3a5a364c2743"
  },
  {
    "case": "5x72_ligand_p59",
    "branches": 2,
    "retained_atoms": 25,
    "fragment_sizes": [
      6,
      7,
      12
    ],
    "prior_bytes_match": true,
    "prior_permutation_matches": true,
    "pdbqt_sha256": "5cb945758d898d05df7e76fa692f181802387e62903a66b95c3519df984dfbf1"
  },
  {
    "case": "5x72_ligan

In [3]:
print(json.dumps(maps, indent=2))

[
  {
    "case": "1iep_ligand",
    "retained_to_source": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      43,
      47,
      58
    ],
    "pdbqt_to_retained": [
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      39,
      27,
      24,
      21,
      22,
      23,
      25,
      26,
      20,
      19,
      35,
      38,
      17,
      13,
      14,
      15,
      16,
      18,
      36,
      12,
      37,
      10,
      6,
      7,
      8,
      9,
      11,
      2,
      0,
      1,
      3,
      4,
      5
    ],
    "pdbqt_to_source": [
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      58,
      27,
     

## Native Vina admission

This checks parser admission of all four generated trees. Real flexible docking
and returned-pose correspondence are separately covered by
`tests/test_flexible_ligand.py`. Successful parsing does not validate chemistry.


In [4]:
import vina

for prepared in prepared_cases:
    vina.Vina(verbosity=0).set_ligand_from_string(prepared.to_pdbqt())
print(json.dumps({"Vina_version": vina.__version__, "generated_ligands_admitted": len(prepared_cases)}, indent=2))

{
  "Vina_version": "1.2.7",
  "generated_ligands_admitted": 4
}
